# Historical experiment notebook
This notebook preserves its original protocol and embedded source. For offline result reproduction use `scripts/reproduce.py` in the repository. For a new measurement keep the original experiment identity and do not overwrite old results. Archive checks are intentionally strict.


# CPU–GPU Research — Step 2
## Memory estimation and a budget-aware placement planner

Step 1 predicts latency. Step 2 adds peak-allocation prediction and chooses a placement under an explicit budget.
**No GPU is required. No inference engine is changed. No new GPT-2 performance measurements are taken.**

You need two files: your completed `step1_..._export.zip` and the original `run_20260919T162736Z_export.zip`.
The latter supplies memory measurements omitted from Step 1's calibration table. Source provenance is checked.

Run Sections 1–7 in order. Stop on a test/audit failure. Retain the original notebooks and results.
This remains retrospective research development, not a blind test or a production admission controller.

## 1. Install the additive extension

The source is embedded below and verified by SHA-256 before extraction. Uploaded ZIP contents are not executed.
The directory is new and timestamped; the previous Step 1 and inference projects are untouched.
Dependencies do not include PyTorch, CUDA or model downloads.

In [ ]:
import os, sys, json, zipfile, base64, hashlib, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
BASE_WORK = Path(os.environ.get("OFFLOAD_STEP2_WORKROOT", "/content" if Path("/content").exists() else str(Path.cwd())))
WORK = BASE_WORK / ("cpu_gpu_research_step2_" + STAMP)
WORK.mkdir(parents=True, exist_ok=False)
SOURCE_B64 = "..."
SOURCE_SHA256 = "43f6e48b43ad86850f202c246f4f47378d10ad795f714a52459bc094322d2e09"
source_bytes = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(source_bytes).hexdigest() == SOURCE_SHA256, "Embedded source hash mismatch"
SOURCE_ZIP = WORK / "step2_source.zip"
SOURCE_ZIP.write_bytes(source_bytes)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for info in z.infolist():
        target = (WORK / info.filename).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise ValueError("Unsafe embedded ZIP path")
    z.extractall(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(WORK / "requirements-research.txt")], check=True)
sys.path.insert(0, str(WORK))
# A fresh notebook avoids stale modules from a previous implementation.
import pandas as pd
from IPython.display import display
OUT = WORK / "results" / ("step2_" + STAMP)
print("Source:", WORK)
print("Checkpoint:", OUT)

## 2. Upload BOTH input archives

Select the Step 1 export and the September 19 full-sweep export together in the upload dialog.
Do not select the original source ZIP or the analysis ZIP. Files are recognized by contents, not their download names.

For local use only, populate both optional paths below or set the corresponding environment variables.

In [ ]:
LOCAL_STEP1_ARCHIVE = os.environ.get("OFFLOAD_STEP1_ARCHIVE", "")
LOCAL_BASELINE_ARCHIVE = os.environ.get("OFFLOAD_BASELINE_ARCHIVE", "")

if LOCAL_STEP1_ARCHIVE and LOCAL_BASELINE_ARCHIVE:
    paths = [Path(LOCAL_STEP1_ARCHIVE), Path(LOCAL_BASELINE_ARCHIVE)]
else:
    from google.colab import files
    print("Select both: step1_..._export.zip AND run_20260919T162736Z_export.zip")
    uploaded = files.upload()
    inbox = WORK / "uploaded_inputs"
    inbox.mkdir(exist_ok=True)
    paths = []
    for name, data in uploaded.items():
        path = inbox / Path(name).name
        path.write_bytes(data)
        paths.append(path)

step1_matches, baseline_matches = [], []
for path in paths:
    if not path.is_file() or not zipfile.is_zipfile(path):
        raise ValueError(f"Not a readable ZIP: {path}")
    with zipfile.ZipFile(path) as z:
        names = set(z.namelist())
    if {"models/compute.json", "models/compute_transfer.json", "calibration.csv", "input_manifest.json"} <= names:
        step1_matches.append(path)
    if any(name.endswith("main/summary.csv") for name in names):
        baseline_matches.append(path)
if len(step1_matches) != 1 or len(baseline_matches) != 1:
    raise ValueError("Upload exactly one completed Step 1 export and one original full-sweep export, then rerun this cell.")
STEP1_ARCHIVE, BASELINE_ARCHIVE = step1_matches[0], baseline_matches[0]
print("Step 1:", STEP1_ARCHIVE.name)
print("Original measurements:", BASELINE_ARCHIVE.name)

## 3. Run software tests

The suite includes the 39 original Step 1 tests plus tests for memory formulas, tied weights, cache sizes,
input checks, shared feasibility filters, fallback, budget violations and regret. Expected: **102 passed**.
These are CPU software tests, not a new pretrained-model or T4 validation run.

In [ ]:
test = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=WORK,
                      text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
TEST_OUTPUT = test.stdout
print(TEST_OUTPUT)
(WORK / "test_output.txt").write_text(TEST_OUTPUT)
test.check_returncode()

## 4. Audit Step 1, fit memory and evaluate policies

The audit rechecks the original 450 timing rows, matches the input hashes, verifies exact parameter/cache accounting,
and reproduces Step 1's grouped predictions and saved models.

The memory prediction is:

`GPU parameter payload + GPU buffers + final GPU KV payload + fitted nonnegative overhead`.

A fixed heuristic margin of **5% + 16 MiB** is added for GPU candidates. It is not a confidence interval or an OOM guarantee.
The budget concerns **peak PyTorch allocated memory**, not total process or device memory.
See [PyTorch's definition](https://docs.pytorch.org/docs/2.11/generated/torch.cuda.memory.max_memory_allocated.html).

Both latency and memory fits withhold a complete workload at a time. Three policies share the same predicted-memory filter:
`max_gpu`, `compute`, and `compute_transfer`. The eight budgets are 256/320/384/448/512/640/768/1024 MiB.
The reference is the best measured feasible candidate among the same five prefix placements, not a globally optimal scheduler.

In [ ]:
command = [sys.executable, "-m", "offload_research.step2", "--step1", str(STEP1_ARCHIVE.resolve()),
           "--baseline", str(BASELINE_ARCHIVE.resolve()), "--out", str(OUT.resolve())]
run = subprocess.run(command, cwd=WORK, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(run.stdout)
if OUT.exists():
    (OUT / "execution_log.txt").write_text(run.stdout)
run.check_returncode()
(OUT / "test_output.txt").write_text(TEST_OUTPUT)
report = json.loads((OUT / "summary.json").read_text())
assert report["configurations"] == 45
assert report["original_trials_audited"] == 450
assert report["policy_decision_records"] == 216
assert report["new_gpu_inference"] is False

## 5. Inspect errors, failures and policy differences

Small memory-prediction error is not a planner speedup. Check whether the predictors choose anything different from max-GPU.
The conservative guard can reject a faster plan that actually fits. Report this opportunity cost instead of reporting only
zero observed violations. Regret is shown only for actually feasible plans; violations and abstentions remain explicit.

Each policy is scored on 9 workload groups × 8 budgets = 72 decisions. These reuse the same historical workloads;
they are not independent new experiments. Read worst regret as well as the median.

In [ ]:
display(pd.DataFrame([report["memory"]]))
display(pd.DataFrame.from_dict(report["policy_metrics"], orient="index"))
print("Choices matching the max-GPU baseline:")
print(json.dumps(report["comparison_to_max_gpu"], indent=2))
policy_rows = pd.read_csv(OUT / "policy_cv.csv")
print("Largest feasible regrets (all policies retained):")
cols = ["policy", "batch_size", "sequence_length", "budget_mib", "selected_gpu_layers",
        "oracle_gpu_layers", "actual_selected_peak_mib", "regret_pct", "status"]
display(policy_rows.sort_values("regret_pct", ascending=False)[cols].head(9))
print("Definitions and limitations are saved in summary.md and protocol.json.")

## 6. Use the planner without running inference

This demonstration uses **batch 3, prompt 64, output 32** and a 400 MiB experimental allocation budget.
It is inside the batch/prompt calibration ranges but is not a measured workload in the original sweep.
All values below are predictions. Hardware/model/backend changes require revalidation.
Only prefix GPU counts 0/3/6/9/12 are supported in this checkpoint; output lengths other than 32 require explicit extrapolation.

In [ ]:
from offload_research.cost_model import CostModel
from offload_research.memory_model import MemoryModel
from offload_research.planner import PlacementPlanner

planner = PlacementPlanner(
    memory=MemoryModel.load(OUT / "models" / "memory.json"),
    compute=CostModel.load(OUT / "models" / "compute.json"),
    compute_transfer=CostModel.load(OUT / "models" / "compute_transfer.json"),
)
plan = planner.recommend(batch_size=3, sequence_length=64, new_tokens=32,
                         gpu_budget_mib=400, policy="compute_transfer")
print(json.dumps(plan, indent=2))
display(planner.candidates(batch_size=3, sequence_length=64, new_tokens=32))
(OUT / "interactive_demo_NOT_MEASURED.json").write_text(json.dumps(plan, indent=2) + "\n")

## 7. Export this checkpoint and stop

The export includes models, predictions, policy outcomes, the audit, source, tests and copies of both original input archives.
Original evidence retains its identity; new predictions are explicitly marked as unmeasured.
No model weights or credentials are included.

Keep the downloaded `step2_..._export.zip`. Review this checkpoint before freezing the code/protocol and collecting fresh GPU tests.

In [ ]:
(OUT / "inputs").mkdir(exist_ok=True)
shutil.copyfile(STEP1_ARCHIVE, OUT / "inputs" / "step1_export.zip")
shutil.copyfile(BASELINE_ARCHIVE, OUT / "inputs" / "baseline_export.zip")
shutil.copyfile(SOURCE_ZIP, OUT / "step2_source.zip")
for name in ["README_STEP2.md", "TEST_STATUS_STEP2.md", "PROTOCOL_STEP2.json"]:
    shutil.copyfile(WORK / name, OUT / name)
(OUT / "analysis_environment.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
archive_path = Path(shutil.make_archive(str(OUT) + "_export", "zip", root_dir=OUT))
print("Exported:", archive_path)
print("No new GPU benchmarks are contained in this checkpoint.")
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Local run: use the ZIP at the printed path.")